# 🎬 hermes-acp-sdk — full showcase (live Hermes)

This notebook was **run against a real Hermes Agent** (`hermes-agent 0.18.2` on DeepSeek)
and is committed **with its output saved** — so you can read the real conversation, the
agent's reasoning and its token usage **without a key and without spending anything**.

**The tour**
1. Connect to a live agent and stream its answer
2. Watch it **think** — the reasoning stream
3. The full event taxonomy
4. The trap this SDK hides (and why nothing works without it)
5. Your own functions as agent tools (MCP)
6. Safe by default

> ℹ️ **This notebook is RECORDED.** The live-Hermes cells below (§1, §2) show their real
> saved output. Re-running them needs `hermes-agent[acp]` **and** a provider key — on the
> shared lab there is neither (by design: no keys, nothing can spend money), so they
> **skip gracefully** instead of erroring. §4–§5 run anywhere, no key, zero tokens.

## 0 · Setup

In [1]:
import logging, os, shutil, sys
from pathlib import Path

# mcp / httpx / uvicorn log at INFO to stderr and Jupyter paints stderr red —
# looks like errors but isn't. Quiet the chatter so the output stays clean.
for _n in ("mcp", "httpx", "uvicorn", "uvicorn.error", "uvicorn.access", "asyncio"):
    logging.getLogger(_n).setLevel(logging.WARNING)

from hermes_acp_sdk import (
    HermesClient, AgentText, AgentThought, ToolCall, PlanUpdated,
    Usage, PermissionDenied, Finished, ToolServer,
)

# convenience: pick provider keys up from a nearby .env
for d in [Path.cwd(), *Path.cwd().parents]:
    f = d / ".env"
    if f.is_file():
        for line in f.read_text().splitlines():
            if "=" in line and not line.strip().startswith("#"):
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
        break

# Hermes auto-detects the provider from whichever key is in the environment.
# This run used a FREE reasoning model on OpenRouter, so the reasoning stream below
# is real. Pass model=... to HermesClient to pin any model the agent advertises.
MODEL = "openrouter:tencent/hy3:free" if os.environ.get("OPENROUTER_API_KEY") else None

_have_hermes = bool(shutil.which("hermes") or (Path(sys.executable).parent / "hermes").exists())
_have_key = any(os.environ.get(k) for k in ("OPENROUTER_API_KEY", "DEEPSEEK_API_KEY", "GEMINI_API_KEY"))
LIVE = _have_hermes and _have_key            # can we actually talk to a live agent?

def recorded_note():
    print("ℹ️  No live Hermes in this environment (hermes:", _have_hermes, "| key:", _have_key, ")")
    print("    The saved output above was recorded against a real one. Install")
    print('    `hermes-agent[acp]` + a provider key to run this live.')

keys = [k for k in ("OPENROUTER_API_KEY", "DEEPSEEK_API_KEY", "GEMINI_API_KEY") if os.environ.get(k)]
print("provider key(s):", ", ".join(keys) or "none")
print("model          :", MODEL or "(agent default)")
print("live Hermes    :", "yes ✅" if LIVE else "no — §1/§2 will show their saved output")

provider key(s): OPENROUTER_API_KEY
model          : openrouter:tencent/hy3:free
live Hermes    : yes ✅


## 1 · Talk to a live Hermes — this is the whole API

`HermesClient()` spawns `hermes acp` and does the handshake. `session()` opens a session
**and selects a model**. `prompt()` is an async iterator, so streaming is a `for` loop.

In [2]:
if not LIVE:
    recorded_note()
else:
    async with HermesClient(model=MODEL) as hermes:
        print(f"connected to: {hermes.agent_name} {hermes.agent_version}\n")

        async with hermes.session() as s:
            async for ev in s.prompt("In one short sentence: what is a Python traceback?"):
                if isinstance(ev, AgentText):
                    print(ev.text, end="", flush=True)          # streams in token by token
                elif isinstance(ev, Finished):
                    print(f"\n\n[finished · stop_reason={ev.stop_reason}]")

connected to: hermes-agent 0.18.2



A

 Python traceback

 is the

 error report

...

 and code)

 that led to

 the error

.



[finished · stop_reason=end_turn]


## 2 · 🧠 Watch it think, and see every event

Everything the agent emits becomes a typed event. You decide what to render: the answer,
the reasoning, tool calls, the plan, token usage.

In [3]:
from collections import Counter

if not LIVE:
    recorded_note()
else:
    from IPython.display import display, HTML

    events = []
    async with HermesClient(model=MODEL) as hermes:
        async with hermes.session() as s:
            # The agent advertises many models — tuck them under a click-to-expand
            # box so they don't flood the notebook. The lab is pinned to a free one.
            _models = "\n".join(m for m, _ in s.available_models)
            display(HTML(
                "<details style='margin:4px 0'>"
                "<summary style='cursor:pointer'>📋 models this agent offers "
                f"({len(s.available_models)}) — click to expand</summary>"
                f"<pre style='margin:6px 0 0'>{_models}</pre></details>"
            ))
            async for ev in s.prompt("Think it through briefly, then answer: what is 12 * 12?"):
                events.append(ev)

    print("events received:", dict(Counter(type(e).__name__ for e in events)), "\n")

    thoughts = "".join(e.text for e in events if isinstance(e, AgentThought))
    answer   = "".join(e.text for e in events if isinstance(e, AgentText))
    usage    = [e for e in events if isinstance(e, Usage)]

    if thoughts:
        print("🧠 THOUGHTS >>>", thoughts[:220].strip(), "…\n")
    else:
        print("🧠 THOUGHTS >>> (this model doesn't stream its reasoning — reasoning models")
        print("               like deepseek-reasoner do, as AgentThought events)\n")
    print("💬 ANSWER   >>>", answer.strip())
    print("📊 USAGE    >>>", usage[-1] if usage else "n/a")

events received: {'Usage': 3, 'AgentText': 1, 'Finished': 1} 

🧠 THOUGHTS >>> (this model doesn't stream its reasoning — reasoning models
               like deepseek-reasoner do, as AgentThought events)

💬 ANSWER   >>> 144
📊 USAGE    >>> Usage(input_tokens=17759, output_tokens=3, total_tokens=17762, used=None, size=None)


## 3 · 🪤 The trap this SDK hides for you

Two things documented **nowhere**, found by driving a real Hermes:

1. **`new_session` lies about the model.** It reports a `current_model_id`, but inference
   still goes out with an **empty** model, so *every* prompt comes back as
   `HTTP 400: The supported API model names are deepseek-v4-pro or deepseek-v4-flash, but you passed .`
   You **must** call `set_session_model` explicitly.
   **`session()` always does it for you** — that is the single biggest reason this package exists.
2. **Never pass `--provider` / `-m`** to `hermes` — those flags *blank out* the model.

The answers you just read are the proof: without the automatic `set_session_model` you
would have got the HTTP 400 blurb instead.

## 4 · 🛠 Give the agent your app's own tools (MCP)

`session(tools=[...])` runs an MCP server **inside this process**, so your functions keep
full access to your app's state. Here we serve one and call it with a real MCP client —
**no LLM involved, zero tokens.**

In [4]:
from mcp.client.session import ClientSession
from mcp.client.streamable_http import streamablehttp_client

ERROR_HISTORY = {"bex": "IndexError (9x), NameError (3x)"}

def student_weakness(student_id: str) -> str:
    "Look up which Python errors a given student most often gets wrong."
    return ERROR_HISTORY.get(student_id, "no history")

async with ToolServer([student_weakness], name="app-tools") as tools:
    cfg = tools.mcp_config
    print("MCP server :", cfg.url, "(loopback + bearer token)")
    headers = {h.name: h.value for h in cfg.headers}
    async with streamablehttp_client(cfg.url, headers=headers) as (r, w, _):
        async with ClientSession(r, w) as mcp:
            await mcp.initialize()
            listed = await mcp.list_tools()
            print("tool       :", listed.tools[0].name, "—", listed.tools[0].description)
            out = await mcp.call_tool("student_weakness", {"student_id": "bex"})
            print("CALL RESULT:", out.content[0].text)

MCP server : http://127.0.0.1:65443/mcp (loopback + bearer token)
tool       : student_weakness — Look up which Python errors a given student most often gets wrong.
CALL RESULT: IndexError (9x), NameError (3x)


A real Hermes picks these up. Its own log during an ACP session reads:

```
MCP server 'app-tools' (HTTP): registered 5 tool(s): mcp__app_tools__student_weakness, ...
refreshed tool surface after ACP MCP registration (28 tools)
```

**This is the piece that lets a coach remember a student:** the agent can pull live
application state — a student's error history — through a function running in *your* process.

## 5 · 🔒 Safe by default *(zero tokens)*

In [5]:
from hermes_acp_sdk import EventHandler, DenyAll, FsPolicy, TerminalPolicy

handler = EventHandler(DenyAll(), FsPolicy(), TerminalPolicy())    # the defaults

for label, call in [
    ("read /etc/passwd", handler.read_text_file(path="/etc/passwd", session_id="s")),
    ("run `rm -rf /`",   handler.create_terminal(command="rm", session_id="s", args=["-rf", "/"])),
]:
    try:
        await call
    except PermissionError as e:
        print(f"❌ agent tried to {label:18} → {e}")

print("\nThese policies govern what the agent asks the CLIENT to do.")
print("They do NOT sandbox Hermes itself — the real boundary is the cwd you hand it.")

❌ agent tried to read /etc/passwd   → read access to '/etc/passwd' is not allowed
❌ agent tried to run `rm -rf /`     → terminal access is not allowed (TerminalPolicy.enabled is False)

These policies govern what the agent asks the CLIENT to do.
They do NOT sandbox Hermes itself — the real boundary is the cwd you hand it.


---
### Summary

`hermes-acp-sdk` turns a two-way protocol with a 12-method callback surface into
**one `async for` loop over typed events** — with the model-selection trap handled,
permissions/filesystem/terminals locked down by default, and your own Python functions
available to the agent as tools.

**No Jupyter required** — see `examples/chat.py` for the same API as a terminal chat app.

`pip install hermes-acp-sdk` · <https://github.com/VoixKz/hermes-acp-sdk>